### Imports

In [16]:
import datasets, openai, instructor, random, asyncio
from pydantic import BaseModel

from asyncio import Semaphore
from tenacity import retry, stop_after_attempt, wait_fixed
from tqdm.asyncio import tqdm_asyncio

### Load data

> Bird rag

cannot find how the data was cleaned but the dataset provides ~1500+ sql snippets that involves ~95 different tables that we can use. 

In [2]:
dataset = datasets.load_dataset("567-labs/bird-rag")["train"]

In [3]:
print(dataset[0])
for item in dataset:
    if item["difficulty"] == "challenging":
        print(item["query"])
        break

{'id': '0', 'query': "SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1", 'difficulty': 'simple'}
SELECT T2.School, T2.DOC FROM frpm AS T1 INNER JOIN schools AS T2 ON T1.CDSCode = T2.CDSCode WHERE T2.FundingType = 'Locally funded' AND (T1.`Enrollment (K-12)` - T1.`Enrollment (Ages 5-17)`) > (SELECT AVG(T3.`Enrollment (K-12)` - T3.`Enrollment (Ages 5-17)`) FROM frpm AS T3 INNER JOIN schools AS T4 ON T3.CDSCode = T4.CDSCode WHERE T4.FundingType = 'Locally funded')


Let's analyze a sample SQL query to understand what kind of synthetic questions we can generate:

> SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1

This query:
1. Calculates the percentage of students receiving free meals
2. Filters to Alameda County schools only
3. Returns the school with the highest percentage

Some relevant natural language questions could be:

- "What school in Alameda County has the highest proportion of students on free meal programs?"
- "Which school has the highest free meal participation rate in Alameda County?"

By generating a dataset of similar questions, we can evaluate how well our retrieval system matches user queries to the appropriate SQL snippets.

## Generating Synthetic Questions

Now let's start generating our synthetic questions. We're going to begin by defining some Pydantic models that represent the format of the data that we're working with.

We're doing so because of the following reasons

1. It helps us to be explicit about the data we're working with 
2. We can use these models with the `instructor` library to obtain structured outputs from our LLM calls


### Format data

In [4]:
class Chunk(BaseModel):
    chunk_id: str
    text: str

In [5]:
dataset[0]

{'id': '0',
 'query': "SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1",
 'difficulty': 'simple'}

In [6]:
sample_chunk = Chunk(chunk_id=dataset[0]['id'], text=dataset[0]['query'])
sample_chunk

Chunk(chunk_id='0', text="SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1")

In [7]:
# This is the synthetic question that we want our model to generate
class Question(BaseModel):
    chain_of_thought: str
    question: str

In [8]:
# This is a single question-chunk pair that we'll be uploading to Braintrust as a dataset later on to be used for benchmarking in `benchmark_retrieval.py`
class ChunkEval(BaseModel):
    chunk_id: str
    question: str
    chunk: str

We're using `instructor` because it handles prompt templating with `jinja` for us and provides validated structured outputs. 

All we need to do is to define a Pydantic model that represents a desired output and the library will handle the rest. 

Remember that we want to generate a question that should either be answerable by the data returned by the SQL snippet directly or with some small tweaks.



In [9]:
client = instructor.from_openai(openai.OpenAI())

In [10]:
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": """
        Generate a hypothetical question that can be answered using the following SQL snippet. 

        SQL Snippet:
        {{ snippet }}

        Rules
        - If there are specific values in the snippet, do not use them directly in the question if possible. 
        - The question should be at most 2 sentences long
        - if necessary, consider making the question more challenging using the following constraint - If there's a time period mentioned in the snippet, modify it slightly (Eg. if the snippet is looking at the entire year, change it to 6 months or 1.5 years)
        - The question must be answerable using the SQL snippet or at most with a small tweak
        """,
        }
    ],
    response_model=Question,
    context={
        "snippet": sample_chunk.text
    }, 
)

print(resp.question)

What is the county in California with the highest percentage of K-12 students receiving free meals over the past year, and how does this reflect on the local school enrollment figures?


In [11]:
sample_chunk.text

"SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1"

> ''What is the highest ratio of free meal counts to total enrollments in K-12 schools for a specific county over a recent semester?'


This is a question which the SQL snippet would be highly relevant for. In order to answer this query, we just need to make two changes

1. add in a new time filter of a recent semester << since it says "recent semester"
2. change the county to a variable << we need county names

### The Diversity Problem

We cannot use the same prompt and expect a diverse set of questions. Therefore we need to introduce slight variations in the prompt to generate questions that are different in wording, intent and content. This is crucial in identifying blindspots in our retrieval system. 

In the example below, we're using the same prompt but introducing randomly chosen constraints at each point. This forces the model to write and generate different questions each time, allowing us to collect a more diverse set of questions. The key here is to really introduce different sources of variation when doing these generations.

### Scaling Up Our Questions

With those points in mind, let's scale our question generation up.

We'll do so by generating a question for each SQL snippet marked as challenging. Since this will be a large number of requests, we're going to be doing so asynchronously with the `asyncio` library.

Additionally, to make sure we stay within our rate limits , we'll be using a semaphore to limit the number of concurrent requests.

We're also making sure that we have a good diversity of questions by randomly selecting a constraint from a set of constraints to make the question more challenging.

In [12]:
# Define Instructor Client
client = instructor.from_openai(openai.AsyncOpenAI())

In [13]:
# Define some constraints to make the question more challenging
constraints = [
    "If there's a time period mentioned in the snippet, modify it slightly (Eg. if the snippet is looking at the entire year, change it to 6 months or 1.5 years)",
    "Add in some irrelevant context (Eg. Add information about the weather, a random event or a backstory that isn't mentioned in the snippet)",
    "Changing the value of the filter (Eg. if the snippet is looking at the results in Canada, change the question to ask about another country or city instead)",
]

In [14]:
@retry(stop=stop_after_attempt(3), wait=wait_fixed(10))
async def generate_questions(chunk: Chunk, sem: Semaphore) -> ChunkEval:
    async with sem:
        coro = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": """
                Generate a hypothetical question that can be answered using the following SQL snippet. 

                SQL Snippet:
                {{ snippet }}

                Rules
                - If there are specific values in the snippet, do not use them directly in the question if possible. 
                - The question should be at most 2 sentences long
                - if necessary, consider making the question more challenging using the following constraint of {{ constraint }}
                - The question must be answerable using the SQL snippet or at most with a small tweak
                """,
                }
            ],
            response_model=Question,
            context={"snippet": chunk.text, "constraint": random.choice(constraints)},
        )
        resp = await asyncio.wait_for(coro, timeout=30)

        return ChunkEval(
            chunk_id=chunk.chunk_id,
            question=resp.question,
            chunk=chunk.text,
        )

In [17]:
sem = Semaphore(10)
dataset = [
    item
    for item in datasets.load_dataset("567-labs/bird-rag")["train"]
    if item["difficulty"] == "challenging"
]
dataset = [Chunk(chunk_id=item["id"], text=item["query"]) for item in dataset]

coros = []

num_samples = 2
for chunk in dataset:
    for _ in range(num_samples):
        coros.append(generate_questions(chunk, sem))

questions: list[ChunkEval] = await tqdm_asyncio.gather(*coros)

/var/folders/y3/yltqmghx7fbd0pyyz9g303wm0000gn/T/ipykernel_3954/493035793.py:9: RuntimeWarning: coroutine 'generate_questions' was never awaited
  coros = []
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 290/290 [01:24<00:00,  3.44it/s]


Now that we've generated our questions, let's take a look at what they look like.

In [18]:
from rich import print


for i in range(2):
    print(
        f"""
    Question: {questions[i].question}

    SQL Snippet: {questions[i].chunk}
    """
    )

Question: Which locally funded schools have a higher difference between K-12 enrollment and enrollment for ages
5-17 compared to the average for other locally funded schools?

    SQL Snippet: SELECT T2.School, T2.DOC FROM frpm AS T1 INNER JOIN schools AS T2 ON T1.CDSCode = T2.CDSCode WHERE
T2.FundingType = 'Locally funded' AND (T1.`Enrollment (K-12)` - T1.`Enrollment (Ages 5-17)`) > (SELECT 
AVG(T3.`Enrollment (K-12)` - T3.`Enrollment (Ages 5-17)`) FROM frpm AS T3 INNER JOIN schools AS T4 ON T3.CDSCode = 
T4.CDSCode WHERE T4.FundingType = 'Locally funded')

Question: Which locally funded schools have an enrollment difference of K-12 and ages 5-17 that exceeds the 
average difference among comparable locally funded schools over the past year?

    SQL Snippet: SELECT T2.School, T2.DOC FROM frpm AS T1 INNER JOIN schools AS T2 ON T1.CDSCode = T2.CDSCode WHERE
T2.FundingType = 'Locally funded' AND (T1.`Enrollment (K-12)` - T1.`Enrollment (Ages 5-17)`) > (SELECT 
AVG(T3.`Enrollment (K-12)` - T3.`Enrollment (Ages 5-17)`) FROM frpm AS T3 INNER JOIN schools AS T4 ON T3.CDSCode = 
T4.CDSCode WHERE T4.FundingType = 'Locally funded')

If we look at both of the generated questions, they're essentially asking about the same thing - that is the schools that are locally funded and have an enrollment difference that's above average. However, the questions are slightly different in wording and intent. The first has a time frame of 1 year while the second one is looking at schools in France specifically. This is a small change but it's enough to create diversity in our questions.

We can scale this up further by adding more constraints and generating more questions. This is crucial in uncovering blindspots in our retrieval system.

## Sharing with the team

If you're working with a team, it's incredibly important to be able to share the results of the experiments with them. In this course, we'll be using [Braintrust](https://www.braintrust.dev/) because it provides an easy way to store private datasets that we can use for benchmarking our retrieval. 

In [19]:
import braintrust

# Initialise Braintrust Dataset
dataset = braintrust.init_dataset(project="Text-2-SQL", name="Bird-Bench-Questions")

# Insert Individual Questions row by row
for question in questions:
    dataset.insert(
        input=question.question,
        expected=[question.chunk],
        metadata={"chunk_id": question.chunk_id, "chunk": question.chunk},
    )

print(dataset.summarize())

DatasetSummary(
    project_name='Text-2-SQL',
    dataset_name='Bird-Bench-Questions',
    project_url='https://www.braintrust.dev/app/None_dipam/p/Text-2-SQL',
    dataset_url='https://www.braintrust.dev/app/None_dipam/p/Text-2-SQL/datasets/Bird-Bench-Questions',
    data_summary=DataSummary(new_records=290, total_records=290)
)

## Conclusion

In this notebook, we covered key metrics like precision and recall, and demonstrated how to generate diverse synthetic questions to benchmark our retrieval system using a Text-2-SQL retrieval system as our example.

In the next notebook, we'll be using these same questions to benchmark different retrieval strategies. This will be followed by Notebook 3 where we'll learn to validate our improvements using bootstrapping and confidence intervals.

Looking ahead to Week 2, we'll leverage these concepts to fine-tune our retrieval models. Using both Cohere's managed re-ranker and the open-source BGE embedding model, we'll see how fine-tuning can improve our recall and MRR metrics on domain-specific queries. 

It's important to note that while synthetic data is valuable for development, it should eventually be augmented with real user queries in production.